# Centre the Slab in the Simulation Cell

In [2]:
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional


@dataclass
class AtomicPositions:
    """One image: list of (species, x, y, z)."""
    atoms: List[Tuple[str, float, float, float]]


def parse_qe_input(path: str) -> Tuple[float, List[AtomicPositions]]:
    """
    Parse a Quantum ESPRESSO input file (single or NEB).
    Returns (Lz, list of AtomicPositions) where Lz is the z-length of the cell in angstrom.
    """
    path = Path(path)
    text = path.read_text()

    # --- Parse CELL_PARAMETERS (angstrom) to get Lz ---
    cell_match = re.search(
        r"CELL_PARAMETERS\s+angstrom\s+\S+\s+\S+\s+\S+\s+\S+\s+\S+\s+\S+\s+(\S+)\s+(\S+)\s+(\S+)",
        text,
        re.DOTALL | re.IGNORECASE,
    )
    if not cell_match:
        raise ValueError("CELL_PARAMETERS angstrom not found")
    # Third row is (a3x, a3y, a3z); for orthogonal z-box Lz = a3z
    a3z = float(cell_match.group(3))
    Lz = abs(a3z)

    # --- Parse atomic positions: NEB (multiple images) or single block ---
    positions_list: List[AtomicPositions] = []

    if "BEGIN_POSITIONS" in text:
        # NEB: split by FIRST_IMAGE / INTERMEDIATE_IMAGE / LAST_IMAGE
        pos_section = re.search(r"BEGIN_POSITIONS(.*?)END_POSITIONS", text, re.DOTALL)
        if not pos_section:
            raise ValueError("BEGIN_POSITIONS ... END_POSITIONS not found")
        block = pos_section.group(1)
        # Each image: IMAGE_MARKER followed by ATOMIC_POSITIONS (angstrom) or angstrom, then lines
        image_pattern = re.compile(
            r"(?:FIRST_IMAGE|INTERMEDIATE_IMAGE|LAST_IMAGE)\s+ATOMIC_POSITIONS\s+(?:\(angstrom\)|angstrom)\s+(.*?)(?=(?:FIRST_IMAGE|INTERMEDIATE_IMAGE|LAST_IMAGE)\s+ATOMIC_POSITIONS|END_POSITIONS|\Z)",
            re.DOTALL | re.IGNORECASE,
        )
        for m in image_pattern.finditer(block):
            pos_text = m.group(1).strip()
            atoms = _parse_atomic_positions_block(pos_text)
            positions_list.append(AtomicPositions(atoms=atoms))
    else:
        # Single ATOMIC_POSITIONS block (accept "(angstrom)" or "angstrom")
        pos_match = re.search(
            r"ATOMIC_POSITIONS\s+(?:\(angstrom\)|angstrom)\s+(.*?)(?=K_POINTS|CELL_PARAMETERS|ATOMIC_SPECIES|/\s*$|\Z)",
            text,
            re.DOTALL | re.IGNORECASE,
        )
        if not pos_match:
            raise ValueError("ATOMIC_POSITIONS (angstrom) or ATOMIC_POSITIONS angstrom not found")
        atoms = _parse_atomic_positions_block(pos_match.group(1).strip())
        positions_list.append(AtomicPositions(atoms=atoms))

    return Lz, positions_list


def _parse_atomic_positions_block(block: str) -> List[Tuple[str, float, float, float]]:
    """Parse lines 'Species  x  y  z' into list of (species, x, y, z)."""
    atoms = []
    for line in block.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) >= 4:
            species = parts[0]
            x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
            atoms.append((species, x, y, z))
    return atoms


def pt_z_center(positions: AtomicPositions) -> float:
    """Z-center of Pt atoms only (slab)."""
    z_vals = [z for sp, _x, _y, z in positions.atoms if sp.strip().upper() == "PT"]
    if not z_vals:
        raise ValueError("No Pt atoms found in this image")
    return sum(z_vals) / len(z_vals)


def center_slab_in_z(Lz: float, positions: AtomicPositions) -> AtomicPositions:
    """
    Translate all atoms so that the Pt slab is centered at Lz/2 along z.
    Returns a new AtomicPositions with shifted z coordinates.
    """
    z0 = pt_z_center(positions)
    target = Lz / 2.0
    shift = target - z0
    new_atoms = [(sp, x, y, z + shift) for sp, x, y, z in positions.atoms]
    return AtomicPositions(atoms=new_atoms)


def format_positions(positions: AtomicPositions) -> str:
    """Format atomic positions as QE ATOMIC_POSITIONS (angstrom) block."""
    lines = ["ATOMIC_POSITIONS (angstrom)"]
    for sp, x, y, z in positions.atoms:
        lines.append(f"   {sp:3}  {x:18.10f}  {y:18.10f}  {z:18.10f}")
    return "\n".join(lines)

In [3]:
# Path to your QE input (single or NEB)
input_file = "4-11b.neb.in"   # change as needed
output_file = "4-11b-centred.neb.in"            # set to "4-11b.neb.centred.in" to write; None = only report

Lz, all_positions = parse_qe_input(input_file)
print(f"Cell z-length Lz = {Lz:.6f} Å")
print(f"Number of position sets (images): {len(all_positions)}")
print()

Cell z-length Lz = 26.789639 Å
Number of position sets (images): 15



In [4]:
target_z = Lz / 2.0
tolerance = 0.01   # Å; consider "centred" if Pt z-center is within this of Lz/2

needs_shift = []
for i, pos in enumerate(all_positions):
    zc = pt_z_center(pos)
    shift = target_z - zc
    ok = abs(shift) <= tolerance
    label = "Image 0 (first)" if i == 0 else ("Image " + str(len(all_positions) - 1) + " (last)" if i == len(all_positions) - 1 else f"Image {i}")
    print(f"{label}: Pt z-center = {zc:.4f} Å  (target {target_z:.4f})  shift = {shift:+.4f} Å  {'✓ centred' if ok else '→ will translate'}")
    if not ok:
        needs_shift.append((i, shift))

Image 0 (first): Pt z-center = 13.3943 Å  (target 13.3948)  shift = +0.0006 Å  ✓ centred
Image 1: Pt z-center = 13.3984 Å  (target 13.3948)  shift = -0.0036 Å  ✓ centred
Image 2: Pt z-center = 13.3851 Å  (target 13.3948)  shift = +0.0097 Å  ✓ centred
Image 3: Pt z-center = 13.3922 Å  (target 13.3948)  shift = +0.0026 Å  ✓ centred
Image 4: Pt z-center = 13.3951 Å  (target 13.3948)  shift = -0.0003 Å  ✓ centred
Image 5: Pt z-center = 13.4311 Å  (target 13.3948)  shift = -0.0362 Å  → will translate
Image 6: Pt z-center = 13.5037 Å  (target 13.3948)  shift = -0.1089 Å  → will translate
Image 7: Pt z-center = 13.5683 Å  (target 13.3948)  shift = -0.1735 Å  → will translate
Image 8: Pt z-center = 13.5966 Å  (target 13.3948)  shift = -0.2018 Å  → will translate
Image 9: Pt z-center = 13.5467 Å  (target 13.3948)  shift = -0.1519 Å  → will translate
Image 10: Pt z-center = 13.4557 Å  (target 13.3948)  shift = -0.0609 Å  → will translate
Image 11: Pt z-center = 13.4164 Å  (target 13.3948)  shift

In [5]:
# Translate all images so the Pt slab is centred at Lz/2
centred_positions = [center_slab_in_z(Lz, pos) for pos in all_positions]
print("Centred positions computed for all images.")

Centred positions computed for all images.


In [6]:
def write_centred_qe_input(
    input_path: str,
    output_path: str,
    Lz: float,
    centred_positions: List[AtomicPositions],
) -> None:
    """Rewrite QE input with ATOMIC_POSITIONS replaced by centred ones."""
    text = Path(input_path).read_text()
    if "BEGIN_POSITIONS" in text:
        # NEB: replace the whole BEGIN_POSITIONS ... END_POSITIONS section
        markers = ["FIRST_IMAGE", "INTERMEDIATE_IMAGE", "LAST_IMAGE"]
        new_blocks = []
        for i, pos in enumerate(centred_positions):
            label = markers[0] if i == 0 else (markers[2] if i == len(centred_positions) - 1 else markers[1])
            new_blocks.append(label + "\n" + format_positions(pos))
        new_section = "BEGIN_POSITIONS\n" + "\n".join(new_blocks) + "\nEND_POSITIONS"
        text = re.sub(r"BEGIN_POSITIONS.*?END_POSITIONS", new_section, text, flags=re.DOTALL)
    else:
        # Single ATOMIC_POSITIONS block
        single = format_positions(centred_positions[0])
        text = re.sub(
            r"ATOMIC_POSITIONS\s+(?:\(angstrom\)|angstrom)\s+.*?(?=K_POINTS|CELL_PARAMETERS|ATOMIC_SPECIES|/\s*$|\Z)",
            single + "\n",
            text,
            count=1,
            flags=re.DOTALL | re.IGNORECASE,
        )
    Path(output_path).write_text(text)
    print(f"Written: {output_path}")


if output_file:
    write_centred_qe_input(input_file, output_file, Lz, centred_positions)
else:
    print("output_file not set; showing centred ATOMIC_POSITIONS for first image only:")
    print(format_positions(centred_positions[0]))

Written: 4-11b-centred.neb.in
